**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# 卷积网络

到目前为止，我们一直使用深层全连接网络，用它们探索不同优化策略和网络架构。全连接网络由于计算效率较高，是很好的实验平台；但在实际应用中，所有最先进的结果基本都使用卷积网络。

首先你将实现卷积网络中会用到的几类层。然后你将用这些层在 CIFAR-10 数据集上训练一个卷积网络。

In [ ]:
# 设置单元。
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.cnn import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient_array, eval_numerical_gradient
from cs231n.layers import *
from cs231n.fast_layers import *
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
  """ returns relative error """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# 加载预处理后的 CIFAR-10 数据。
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# 卷积：朴素前向传播
卷积网络的核心是卷积操作。在 `cs231n/layers.py` 文件中，在 `conv_forward_naive` 函数里实现卷积层的前向传播。

现在不需要太担心效率；用你觉得最清晰的方式写代码即可。

运行下面的代码可以测试你的实现：

In [ ]:
x_shape = (2, 3, 4, 4)
w_shape = (3, 3, 4, 4)
x = np.linspace(-0.1, 0.5, num=np.prod(x_shape)).reshape(x_shape)
w = np.linspace(-0.2, 0.3, num=np.prod(w_shape)).reshape(w_shape)
b = np.linspace(-0.1, 0.2, num=3)

conv_param = {'stride': 2, 'pad': 1}
out, _ = conv_forward_naive(x, w, b, conv_param)
correct_out = np.array([[[[-0.08759809, -0.10987781],
                           [-0.18387192, -0.2109216 ]],
                          [[ 0.21027089,  0.21661097],
                           [ 0.22847626,  0.23004637]],
                          [[ 0.50813986,  0.54309974],
                           [ 0.64082444,  0.67101435]]],
                         [[[-0.98053589, -1.03143541],
                           [-1.19128892, -1.24695841]],
                          [[ 0.69108355,  0.66880383],
                           [ 0.59480972,  0.56776003]],
                          [[ 2.36270298,  2.36904306],
                           [ 2.38090835,  2.38247847]]]])

# Compare your 输出 到 ours; difference 应为 约为 e-8
print('Testing conv_forward_naive')
print('difference: ', rel_error(out, correct_out))

## 插曲：用卷积做图像处理

为了检查你的实现，并更好理解卷积层能执行哪些类型的操作，我们会构造一个包含两张图像的输入，并手动设置一些滤波器来执行常见的图像处理操作（灰度转换和边缘检测）。卷积前向传播会把这些操作应用到每张输入图像上。随后我们可以可视化结果，作为合理性检查。

In [ ]:
from imageio import imread
from PIL import Image

kitten = imread('cs231n/notebook_images/kitten.jpg')
puppy = imread('cs231n/notebook_images/puppy.jpg')
# kitten is wide, 并 puppy is already square
d = kitten.shape[1] - kitten.shape[0]
kitten_cropped = kitten[:, d//2:-d//2, :]

img_size = 200   # Make 这个 更小 if it runs too slow
resized_puppy = np.array(Image.fromarray(puppy).resize((img_size, img_size)))
resized_kitten = np.array(Image.fromarray(kitten_cropped).resize((img_size, img_size)))
x = np.zeros((2, 3, img_size, img_size))
x[0, :, :, :] = resized_puppy.transpose((2, 0, 1))
x[1, :, :, :] = resized_kitten.transpose((2, 0, 1))

# 设置包含 2 个 3x3 滤波器的卷积权重
w = np.zeros((2, 3, 3, 3))

# 第一个滤波器将图像转换为灰度。
# 设置滤波器的红、绿、蓝通道。
w[0, 0, :, :] = [[0, 0, 0], [0, 0.3, 0], [0, 0, 0]]
w[0, 1, :, :] = [[0, 0, 0], [0, 0.6, 0], [0, 0, 0]]
w[0, 2, :, :] = [[0, 0, 0], [0, 0.1, 0], [0, 0, 0]]

# 第二个滤波器检测蓝色通道中的水平边缘。
w[1, 2, :, :] = [[1, 2, 1], [0, 0, 0], [-1, -2, -1]]

# bias 向量。灰度滤波器不需要 bias
# filter, but 用于 edge detection filter we want 到 add 128
# to each 输出 so 该 nothing is negative.
b = np.array([0, 128])

# 计算 结果 的 convolving each 输入 在 x 使用 each filter 在 w,
# offsetting by b, 并 将结果存储s 在 out.
out, _ = conv_forward_naive(x, w, b, {'stride': 1, 'pad': 1})

def imshow_no_ax(img, normalize=True):
    """ Tiny helper 到 show images as uint8 并 remove axis 标签 """
    if normalize:
        img_max, img_min = np.max(img), np.min(img)
        img = 255.0 * (img - img_min) / (img_max - img_min)
    plt.imshow(img.astype('uint8'))
    plt.gca().axis('off')

# 显示原始图像和卷积操作结果
plt.subplot(2, 3, 1)
imshow_no_ax(puppy, normalize=False)
plt.title('Original image')
plt.subplot(2, 3, 2)
imshow_no_ax(out[0, 0])
plt.title('Grayscale')
plt.subplot(2, 3, 3)
imshow_no_ax(out[0, 1])
plt.title('Edges')
plt.subplot(2, 3, 4)
imshow_no_ax(kitten_cropped, normalize=False)
plt.subplot(2, 3, 5)
imshow_no_ax(out[1, 0])
plt.subplot(2, 3, 6)
imshow_no_ax(out[1, 1])
plt.show()

# 卷积：朴素反向传播
在 `cs231n/layers.py` 文件的 `conv_backward_naive` 函数中实现卷积操作的反向传播。同样，这里不需要太担心计算效率。

完成后，运行下面的代码，用数值梯度检查你的反向传播。

_提示：https://deeplearning.cs.cmu.edu/F21/document/recitation/Recitation5/CNN_Backprop_Recitation_5_F21.pdf 可以帮助你获得总体理解。实际的朴素实现会使用嵌套 for 循环，并利用来自 `dw` 和 `dout` 的单独导数直接计算 `dx_padded`。_

In [ ]:
np.random.seed(231)
x = np.random.randn(4, 3, 5, 5)
w = np.random.randn(2, 3, 3, 3)
b = np.random.randn(2,)
dout = np.random.randn(4, 2, 5, 5)
conv_param = {'stride': 1, 'pad': 1}

dx_num = eval_numerical_gradient_array(lambda x: conv_forward_naive(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_forward_naive(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_forward_naive(x, w, b, conv_param)[0], b, dout)

out, cache = conv_forward_naive(x, w, b, conv_param)
dx, dw, db = conv_backward_naive(dout, cache)

# Your errors 应为 约为 e-8 or less.
print('Testing conv_backward_naive function')
print('dx error: ', rel_error(dx, dx_num))
print('dw error: ', rel_error(dw, dw_num))
print('db error: ', rel_error(db, db_num))

# Max-Pooling：朴素前向传播
在 `cs231n/layers.py` 文件的 `max_pool_forward_naive` 函数中实现 max-pooling 操作的前向传播。同样，不必太担心计算效率。

运行下面的代码检查你的实现：

In [ ]:
x_shape = (2, 3, 4, 4)
x = np.linspace(-0.3, 0.4, num=np.prod(x_shape)).reshape(x_shape)
pool_param = {'pool_width': 2, 'pool_height': 2, 'stride': 2}

out, _ = max_pool_forward_naive(x, pool_param)

correct_out = np.array([[[[-0.26315789, -0.24842105],
                          [-0.20421053, -0.18947368]],
                         [[-0.14526316, -0.13052632],
                          [-0.08631579, -0.07157895]],
                         [[-0.02736842, -0.01263158],
                          [ 0.03157895,  0.04631579]]],
                        [[[ 0.09052632,  0.10526316],
                          [ 0.14947368,  0.16421053]],
                         [[ 0.20842105,  0.22315789],
                          [ 0.26736842,  0.28210526]],
                         [[ 0.32631579,  0.34105263],
                          [ 0.38526316,  0.4       ]]]])

# Compare your 输出 使用 ours. Difference 应为 on order 的 e-8.
print('Testing max_pool_forward_naive function:')
print('difference: ', rel_error(out, correct_out))

# Max-Pooling：朴素反向传播
在 `cs231n/layers.py` 文件的 `max_pool_backward_naive` 函数中实现 max-pooling 操作的反向传播。不需要担心计算效率。

运行下面的代码，用数值梯度检查你的实现：

In [ ]:
np.random.seed(231)
x = np.random.randn(3, 2, 8, 8)
dout = np.random.randn(3, 2, 4, 4)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

dx_num = eval_numerical_gradient_array(lambda x: max_pool_forward_naive(x, pool_param)[0], x, dout)

out, cache = max_pool_forward_naive(x, pool_param)
dx = max_pool_backward_naive(dout, cache)

# Your error 应为 on order 的 e-12
print('Testing max_pool_backward_naive function:')
print('dx error: ', rel_error(dx, dx_num))

# 快速层

让卷积层和池化层运行得很快并不容易。为减少这部分痛苦，我们已经在 `cs231n/fast_layers.py` 中提供了卷积和池化层前向、反向传播的快速实现。

### 执行下面的单元，保存 notebook，然后重启 runtime
快速卷积实现依赖一个 Cython 扩展；要编译它，请运行下面的单元。接着保存 Colab notebook（`File > Save`）并**重启 runtime**（`Runtime > Restart runtime`）。之后你可以从上到下重新执行前面的单元，并跳过下面这个单元，因为编译步骤只需要运行一次。

In [ ]:
# 执行这个单元后，请记得重启 runtime。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/
!python setup.py build_ext --inplace
%cd /content/drive/My\ Drive/$FOLDERNAME/

快速版卷积层和池化层的 API 与你上面实现的朴素版本完全相同：前向传播接收数据、权重和参数，产生输出和 cache 对象；反向传播接收上游导数和 cache 对象，产生关于数据和权重的梯度。

**注意：** 快速版 pooling 只有在 pooling 区域互不重叠且正好铺满输入时才会达到最优性能。如果不满足这些条件，快速 pooling 实现不会比朴素实现快太多。

运行下面的代码可以比较这些层朴素版本和快速版本的性能：

In [ ]:
# Rel errors 应为 约为 e-9 or less.
from cs231n.fast_layers import conv_forward_fast, conv_backward_fast
from time import time
np.random.seed(231)
x = np.random.randn(100, 3, 31, 31)
w = np.random.randn(25, 3, 3, 3)
b = np.random.randn(25,)
dout = np.random.randn(100, 25, 16, 16)
conv_param = {'stride': 2, 'pad': 1}

t0 = time()
out_naive, cache_naive = conv_forward_naive(x, w, b, conv_param)
t1 = time()
out_fast, cache_fast = conv_forward_fast(x, w, b, conv_param)
t2 = time()

print('Testing conv_forward_fast:')
print('Naive: %fs' % (t1 - t0))
print('Fast: %fs' % (t2 - t1))
print('Speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('Difference: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive, dw_naive, db_naive = conv_backward_naive(dout, cache_naive)
t1 = time()
dx_fast, dw_fast, db_fast = conv_backward_fast(dout, cache_fast)
t2 = time()

print('\nTesting conv_backward_fast:')
print('Naive: %fs' % (t1 - t0))
print('Fast: %fs' % (t2 - t1))
print('Speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('dx difference: ', rel_error(dx_naive, dx_fast))
print('dw difference: ', rel_error(dw_naive, dw_fast))
print('db difference: ', rel_error(db_naive, db_fast))

In [ ]:
# Relative errors 应为 接近 0.0.
from cs231n.fast_layers import max_pool_forward_fast, max_pool_backward_fast
np.random.seed(231)
x = np.random.randn(100, 3, 32, 32)
dout = np.random.randn(100, 3, 16, 16)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

t0 = time()
out_naive, cache_naive = max_pool_forward_naive(x, pool_param)
t1 = time()
out_fast, cache_fast = max_pool_forward_fast(x, pool_param)
t2 = time()

print('Testing pool_forward_fast:')
print('Naive: %fs' % (t1 - t0))
print('fast: %fs' % (t2 - t1))
print('speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('difference: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive = max_pool_backward_naive(dout, cache_naive)
t1 = time()
dx_fast = max_pool_backward_fast(dout, cache_fast)
t2 = time()

print('\nTesting pool_backward_fast:')
print('Naive: %fs' % (t1 - t0))
print('fast: %fs' % (t2 - t1))
print('speedup: %fx' % ((t1 - t0) / (t2 - t1)))
print('dx difference: ', rel_error(dx_naive, dx_fast))

# 卷积 “三明治” 层
在上一个作业中，我们介绍了 “sandwich” 层的概念，它们把多个操作组合成常见模式。在 `cs231n/layer_utils.py` 文件中，你会找到一些为卷积网络常用模式实现的 sandwich 层。运行下面的单元，对它们的用法做合理性检查。

In [ ]:
from cs231n.layer_utils import conv_relu_pool_forward, conv_relu_pool_backward
np.random.seed(231)
x = np.random.randn(2, 3, 16, 16)
w = np.random.randn(3, 3, 3, 3)
b = np.random.randn(3,)
dout = np.random.randn(2, 3, 8, 8)
conv_param = {'stride': 1, 'pad': 1}
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

out, cache = conv_relu_pool_forward(x, w, b, conv_param, pool_param)
dx, dw, db = conv_relu_pool_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], b, dout)

# Relative errors 应为 约为 e-8 or less
print('Testing conv_relu_pool')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

In [ ]:
from cs231n.layer_utils import conv_relu_forward, conv_relu_backward
np.random.seed(231)
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(3, 3, 3, 3)
b = np.random.randn(3,)
dout = np.random.randn(2, 3, 8, 8)
conv_param = {'stride': 1, 'pad': 1}

out, cache = conv_relu_forward(x, w, b, conv_param)
dx, dw, db = conv_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_forward(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_forward(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_forward(x, w, b, conv_param)[0], b, dout)

# Relative errors 应为 约为 e-8 or less
print('Testing conv_relu:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# 三层卷积网络
现在你已经实现了所有必要的层，可以把它们组合成一个简单的卷积网络。

打开 `cs231n/classifiers/cnn.py` 文件，完成 `ThreeLayerConvNet` 类的实现。记住，你可以在实现中使用 fast/sandwich 层（已经为你导入）。运行下面的单元帮助你调试：

## 合理性检查：损失
构建新网络后，首先应该检查损失是否合理。使用 softmax 损失时，随机权重（且无正则化）对应的损失大约应为 `log(C)`，其中 `C` 是类别数。加入正则化后，损失应该略有上升。

In [ ]:
model = ThreeLayerConvNet()

N = 50
X = np.random.randn(N, 3, 32, 32)
y = np.random.randint(10, size=N)

loss, grads = model.loss(X, y)
print('Initial loss (no regularization): ', loss)

model.reg = 0.5
loss, grads = model.loss(X, y)
print('Initial loss (with regularization): ', loss)

## 梯度检查
当损失看起来合理后，使用数值梯度检查确认反向传播是否正确。做数值梯度检查时，应该使用少量人工数据，并在每层使用较少神经元。注意：正确实现的相对误差仍可能达到 `1e-2` 数量级。

In [ ]:
num_inputs = 2
input_dim = (3, 16, 16)
reg = 0.0
num_classes = 10
np.random.seed(231)
X = np.random.randn(num_inputs, *input_dim)
y = np.random.randint(num_classes, size=num_inputs)

model = ThreeLayerConvNet(
    num_filters=3,
    filter_size=3,
    input_dim=input_dim,
    hidden_dim=7,
    dtype=np.float64
)
loss, grads = model.loss(X, y)
# Errors 应为 sm所有, but correct 实现s may have
# 相对误差s up 到 order 的 e-2
for param_name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    param_grad_num = eval_numerical_gradient(f, model.params[param_name], verbose=False, h=1e-6)
    e = rel_error(param_grad_num, grads[param_name])
    print('%s max relative error: %e' % (param_name, rel_error(param_grad_num, grads[param_name])))

## 在小数据上过拟合
一个有用的技巧是只用少量训练样本训练模型。你应该能够让模型过拟合小数据集，这会带来很高的训练准确率和相对较低的验证准确率。

In [ ]:
np.random.seed(231)

num_train = 100
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

model = ThreeLayerConvNet(weight_scale=1e-2)

solver = Solver(
    model,
    small_data,
    num_epochs=15,
    batch_size=50,
    update_rule='adam',
    optim_config={'learning_rate': 1e-3,},
    verbose=True,
    print_every=1
)
solver.train()

In [ ]:
# 打印最终训练准确率。
print(
    "Small data training accuracy:",
    solver.check_accuracy(small_data['X_train'], small_data['y_train'])
)

In [ ]:
# 打印最终验证准确率。
print(
    "Small data validation accuracy:",
    solver.check_accuracy(small_data['X_val'], small_data['y_val'])
)

绘制损失、训练准确率和验证准确率后，应该能清楚看到过拟合现象：

In [ ]:
plt.subplot(2, 1, 1)
plt.plot(solver.loss_history, 'o')
plt.xlabel('iteration')
plt.ylabel('loss')

plt.subplot(2, 1, 2)
plt.plot(solver.train_acc_history, '-o')
plt.plot(solver.val_acc_history, '-o')
plt.legend(['train', 'val'], loc='upper left')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.show()

## 训练网络
训练三层卷积网络一个 epoch 后，你应该在训练集上达到高于 40% 的准确率：

In [ ]:
model = ThreeLayerConvNet(weight_scale=0.001, hidden_dim=500, reg=0.001)

solver = Solver(
    model,
    data,
    num_epochs=1,
    batch_size=50,
    update_rule='adam',
    optim_config={'learning_rate': 1e-3,},
    verbose=True,
    print_every=20
)
solver.train()

In [ ]:
# 打印最终训练准确率。
print(
    "Full data training accuracy:",
    solver.check_accuracy(data['X_train'], data['y_train'])
)

In [ ]:
# 打印最终验证准确率。
print(
    "Full data validation accuracy:",
    solver.check_accuracy(data['X_val'], data['y_val'])
)

## 可视化滤波器
运行下面的代码，可以可视化训练后网络第一层的卷积滤波器：

In [ ]:
from cs231n.vis_utils import visualize_grid

grid = visualize_grid(model.params['W1'].transpose(0, 2, 3, 1))
plt.imshow(grid.astype('uint8'))
plt.axis('off')
plt.gcf().set_size_inches(5, 5)
plt.show()

# Spatial Batch Normalization
我们已经看到 batch normalization 对训练深层全连接网络非常有用。正如原论文中提出的那样（链接见 `BatchNormalization.ipynb`），batch normalization 也可以用于卷积网络，但需要稍作调整；这个变体称为 “spatial batch normalization”。

普通 batch normalization 接收形状为 `(N, D)` 的输入，并输出形状为 `(N, D)` 的结果，其中我们沿 minibatch 维度 `N` 做归一化。对于来自卷积层的数据，batch normalization 需要接收形状为 `(N, C, H, W)` 的输入，并输出同样形状的结果；其中 `N` 表示 minibatch 大小，`(H, W)` 表示特征图的空间尺寸。

如果特征图由卷积产生，那么我们预期每个特征通道的统计量（例如均值和方差）在不同图像之间、同一图像的不同位置之间都相对一致。毕竟，每个特征通道都由同一个卷积滤波器产生。因此，spatial batch normalization 会对每个 `C` 特征通道，沿 minibatch 维度 `N` 以及空间维度 `H` 和 `W` 共同计算统计量。

[1] [Sergey Ioffe and Christian Szegedy, "Batch Normalization: Accelerating Deep Network Training by Reducing
Internal Covariate Shift", ICML 2015.](https://arxiv.org/abs/1502.03167)

# Spatial Batch Normalization：前向传播

在 `cs231n/layers.py` 文件中，在 `spatial_batchnorm_forward` 函数里实现 spatial batch normalization 的前向传播。运行下面的代码检查你的实现：

In [ ]:
np.random.seed(231)

# Check 训练时 前向传播 by checking 均值 并 方差
# of 特征 both before 并 after spatial batch 归一化.
N, C, H, W = 2, 3, 4, 5
x = 4 * np.random.randn(N, C, H, W) + 10

print('Before spatial batch normalization:')
print('  shape: ', x.shape)
print('  means: ', x.mean(axis=(0, 2, 3)))
print('  stds: ', x.std(axis=(0, 2, 3)))

# Means 应为 接近 zero 并 标准差 接近 one
gamma, beta = np.ones(C), np.zeros(C)
bn_param = {'mode': 'train'}
out, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)
print('After spatial batch normalization:')
print('  shape: ', out.shape)
print('  means: ', out.mean(axis=(0, 2, 3)))
print('  stds: ', out.std(axis=(0, 2, 3)))

# Means 应为 接近 beta 并 标准差 接近 gamma
gamma, beta = np.asarray([3, 4, 5]), np.asarray([6, 7, 8])
out, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)
print('After spatial batch normalization (nontrivial gamma, beta):')
print('  shape: ', out.shape)
print('  means: ', out.mean(axis=(0, 2, 3)))
print('  stds: ', out.std(axis=(0, 2, 3)))

In [ ]:
np.random.seed(231)

# Check 测试时 前向传播 by running 训练时
# 前向传播 many times 到 warm up running averages, 并 然后
# checking 均值 并 方差 的 activations after a 测试时
# 前向传播.
N, C, H, W = 10, 4, 11, 12

bn_param = {'mode': 'train'}
gamma = np.ones(C)
beta = np.zeros(C)
for t in range(50):
  x = 2.3 * np.random.randn(N, C, H, W) + 13
  spatial_batchnorm_forward(x, gamma, beta, bn_param)
bn_param['mode'] = 'test'
x = 2.3 * np.random.randn(N, C, H, W) + 13
a_norm, _ = spatial_batchnorm_forward(x, gamma, beta, bn_param)

# Means 应为 接近 zero 并 标准差 接近 one, but 将 be
# noisier than 训练时 前向传播es.
print('After spatial batch normalization (test-time):')
print('  means: ', a_norm.mean(axis=(0, 2, 3)))
print('  stds: ', a_norm.std(axis=(0, 2, 3)))

# Spatial Batch Normalization：反向传播
在 `cs231n/layers.py` 文件中，在 `spatial_batchnorm_backward` 函数里实现 spatial batch normalization 的反向传播。运行下面的代码，用数值梯度检查你的实现：

In [ ]:
np.random.seed(231)
N, C, H, W = 2, 3, 4, 5
x = 5 * np.random.randn(N, C, H, W) + 12
gamma = np.random.randn(C)
beta = np.random.randn(C)
dout = np.random.randn(N, C, H, W)

bn_param = {'mode': 'train'}
fx = lambda x: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]
fg = lambda a: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]
fb = lambda b: spatial_batchnorm_forward(x, gamma, beta, bn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma, dout)
db_num = eval_numerical_gradient_array(fb, beta, dout)

#你应该 expect errors 的 magnitudes between 1e-12~1e-06
_, cache = spatial_batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = spatial_batchnorm_backward(dout, cache)
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))

# Spatial Group Normalization
在前一个 notebook 中，我们提到 Layer Normalization 是一种可以缓解 Batch Normalization 受 batch size 限制影响的替代归一化技术。然而，正如文献 [2] 的作者观察到的那样，把 Layer Normalization 用在卷积层上时，它的表现不如 Batch Normalization：

> 在全连接层中，同一层的所有隐藏单元往往对最终预测作出相似贡献，因此对这一层的求和输入重新中心化和重新缩放通常效果不错。然而，在卷积神经网络中，“贡献相似” 的假设不再成立。大量位于图像边界附近感受野内的隐藏单元很少被激活，因此它们的统计量与同一层中其余隐藏单元非常不同。

文献 [3] 的作者提出了一种折中技术。与 Layer Normalization 对每个数据点的整个特征做归一化不同，他们建议把每个数据点的特征稳定地划分为 `G` 个组，并对每个数据点的每个组分别归一化。

<p align="center">
<img src="https://raw.githubusercontent.com/cs231n/cs231n.github.io/master/assets/a2/normalization.png">
</p>
<center>目前讨论过的归一化技术的视觉对比（图片由 [3] 修改而来）</center>

尽管每个组内仍然假设贡献相似，但作者认为问题没那么严重，因为视觉识别特征中天然存在分组结构。他们用来说明这一点的一个例子是：传统计算机视觉中很多高性能手工特征都会显式分组。例如方向梯度直方图（Histogram of Oriented Gradients, HOG）[4]：在每个局部空间块中计算直方图后，每个块的直方图会先归一化，再拼接成最终特征向量。

现在你将实现 Group Normalization。

[2] [Ba, Jimmy Lei, Jamie Ryan Kiros, and Geoffrey E. Hinton. "Layer Normalization." stat 1050 (2016): 21.](https://arxiv.org/pdf/1607.06450.pdf)

[3] [Wu, Yuxin, and Kaiming He. "Group Normalization." arXiv preprint arXiv:1803.08494 (2018).](https://arxiv.org/abs/1803.08494)

[4] [N. Dalal and B. Triggs. Histograms of oriented gradients for
human detection. In Computer Vision and Pattern Recognition
(CVPR), 2005.](https://ieeexplore.ieee.org/abstract/document/1467360/)

# Spatial Group Normalization：前向传播

在 `cs231n/layers.py` 文件中，在 `spatial_groupnorm_forward` 函数里实现 group normalization 的前向传播。运行下面的代码检查你的实现：

In [ ]:
np.random.seed(231)

# Check 训练时 前向传播 by checking 均值 并 方差
# of 特征 both before 并 after spatial batch 归一化.
N, C, H, W = 2, 6, 4, 5
G = 2
x = 4 * np.random.randn(N, C, H, W) + 10
x_g = x.reshape((N*G,-1))
print('Before spatial group normalization:')
print('  shape: ', x.shape)
print('  means: ', x_g.mean(axis=1))
print('  stds: ', x_g.std(axis=1))

# Means 应为 接近 zero 并 标准差 接近 one
gamma, beta = np.ones((1,C,1,1)), np.zeros((1,C,1,1))
bn_param = {'mode': 'train'}

out, _ = spatial_groupnorm_forward(x, gamma, beta, G, bn_param)
out_g = out.reshape((N*G,-1))
print('After spatial group normalization:')
print('  shape: ', out.shape)
print('  means: ', out_g.mean(axis=1))
print('  stds: ', out_g.std(axis=1))

# Spatial Group Normalization：反向传播
在 `cs231n/layers.py` 文件中，在 `spatial_groupnorm_backward` 函数里实现 spatial group normalization 的反向传播。运行下面的代码，用数值梯度检查你的实现：

In [ ]:
np.random.seed(231)
N, C, H, W = 2, 6, 4, 5
G = 2
x = 5 * np.random.randn(N, C, H, W) + 12
gamma = np.random.randn(1,C,1,1)
beta = np.random.randn(1,C,1,1)
dout = np.random.randn(N, C, H, W)

gn_param = {}
fx = lambda x: spatial_groupnorm_forward(x, gamma, beta, G, gn_param)[0]
fg = lambda a: spatial_groupnorm_forward(x, gamma, beta, G, gn_param)[0]
fb = lambda b: spatial_groupnorm_forward(x, gamma, beta, G, gn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma, dout)
db_num = eval_numerical_gradient_array(fb, beta, dout)

_, cache = spatial_groupnorm_forward(x, gamma, beta, G, gn_param)
dx, dgamma, dbeta = spatial_groupnorm_backward(dout, cache)

# 你应该 expect errors 的 magnitudes between 1e-12 并 1e-07.
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))